# V-JEPA 2.1: inspect dense video features

## Learning outcome

Load the smallest released V-JEPA 2.1 encoder, view its space and time feature grid, and measure one known patch match across a generated video.

## What this lesson does

This is a local feature-inspection exercise on a deterministic synthetic clip. It is **not a reproduction** of the V-JEPA 2.1 paper's benchmark results. The paper reports results on real video, perception, and robotics tasks; this notebook reports only what this local run measures.

The model input uses the public 384-pixel ViT-B/16 entrypoint with 16 frames. The model groups each pair of frames into one temporal tubelet, so the output is expected to contain 8 time groups and a 24×24 patch grid.

## Optional model download

The official checkpoint is about 1.66 GB. No checkpoint or code is downloaded unless you set `JEPA_LOAD_VJEPA21=1` before launching Jupyter. The checkpoint terms are not stated separately in the [pinned official README](https://github.com/facebookresearch/vjepa2/blob/204698b45b3712590f06245fbfba32d3be539812/README.md). Check the current terms at the [checkpoint source](https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt) before you download it. Do not redistribute the weights.

The source code revision and model are pinned below. The official public entrypoint defaults to a localhost checkpoint URL, so this notebook disables its built-in download and loads the published checkpoint URL explicitly.

## Requirements

- Python 3.10–3.13
- The repository's general notebook dependencies
- For the model path: the optional `vjepa21` extra and PyTorch installed from the correct wheel index
- CPU works when local time and memory allow; CUDA is used when available

See [the feature quickstart](../specs/002-vjepa-21-dense-features/quickstart.md) for installation and run commands.

## Research sources

- [V-JEPA 2.1 paper](https://arxiv.org/abs/2603.14482)
- [Official code and checkpoints](https://github.com/facebookresearch/vjepa2)
- [Pinned Hub encoder source](https://github.com/facebookresearch/vjepa2/blob/204698b45b3712590f06245fbfba32d3be539812/src/hub/backbones.py)
- [Pinned preprocessing source](https://github.com/facebookresearch/vjepa2/blob/204698b45b3712590f06245fbfba32d3be539812/evals/hub/preprocessor.py)


In [ ]:
import gc
import hashlib
import json
import os
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

FRAME_COUNT = 16
FRAME_SIZE = 384
PATCH_SIZE = 16
TUBELET_SIZE = 2
TIME_GROUPS = FRAME_COUNT // TUBELET_SIZE
PATCH_ROWS = FRAME_SIZE // PATCH_SIZE
PATCH_COLUMNS = FRAME_SIZE // PATCH_SIZE
SEED = 21

CODE_REVISION = "204698b45b3712590f06245fbfba32d3be539812"
MODEL_ENTRYPOINT = "vjepa2_1_vit_base_384"
CHECKPOINT_NAME = "vjepa2_1_vitb_dist_vitG_384.pt"
CHECKPOINT_URL = f"https://dl.fbaipublicfiles.com/vjepa2/{CHECKPOINT_NAME}"
EXPECTED_CHECKPOINT_BYTES = 1_664_223_428
MODEL_REPO = f"facebookresearch/vjepa2:{CODE_REVISION}"

LOAD_OFFICIAL_MODEL = os.environ.get("JEPA_LOAD_VJEPA21", "").strip().lower() in {"1", "true", "yes"}


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


REPO_ROOT = find_repo_root()
# Load this standard-library module directly so the synthetic lesson stays usable without PyTorch.
from runpy import run_path
CHECKPOINTS_DIR = run_path(str(REPO_ROOT / "src" / "jepa_tutorial" / "storage.py"))["CHECKPOINTS_DIR"]
CHECKPOINT_DIR = CHECKPOINTS_DIR / "vjepa21"
CHECKPOINT_PATH = CHECKPOINT_DIR / CHECKPOINT_NAME

encoder = None
preprocessor = None
device = None
video_tensor = None
feature_grid = None
run_metadata = {}

print(f"Python: {platform.python_version()}")
print(f"Operating system: {platform.platform()}")
print(f"Model download enabled: {LOAD_OFFICIAL_MODEL}")
print(f"Synthetic clip: {FRAME_COUNT} frames, {FRAME_SIZE}×{FRAME_SIZE} pixels")
print(f"Expected grid: {TIME_GROUPS} time groups × {PATCH_ROWS} × {PATCH_COLUMNS} patches")
if not LOAD_OFFICIAL_MODEL:
    print("No model or video download will occur. Set JEPA_LOAD_VJEPA21=1 to opt in to the official model path.")


## 1. Make a deterministic moving-object clip

Each frame contains the same small color pattern at a known position. This gives us a target position without a dataset or labels. The pattern and motion are for inspection only; they are not a video-tracking benchmark.


In [ ]:
tile_palette = np.array(
    [
        [220, 70, 65], [240, 170, 55], [80, 180, 110], [65, 145, 225],
        [170, 90, 210], [65, 205, 200], [235, 115, 170], [185, 205, 65],
        [245, 205, 125], [100, 115, 225], [110, 205, 145], [230, 105, 85],
        [125, 190, 225], [205, 125, 65], [145, 95, 170], [220, 220, 220],
    ],
    dtype=np.uint8,
)


def generate_clip():
    frames = []
    centers_xy = []
    object_size = 64
    tile_size = PATCH_SIZE

    for frame_index in range(FRAME_COUNT):
        frame = np.empty((FRAME_SIZE, FRAME_SIZE, 3), dtype=np.uint8)
        frame[:] = (25, 37, 52)
        frame[:, ::32, :] = (33, 48, 64)

        center_x = int(round(72 + frame_index * (240 / (FRAME_COUNT - 1))))
        center_y = int(round(192 + 18 * np.sin(2 * np.pi * frame_index / (FRAME_COUNT - 1))))
        left = center_x - object_size // 2
        top = center_y - object_size // 2

        for tile_row in range(object_size // tile_size):
            for tile_column in range(object_size // tile_size):
                tile_index = tile_row * (object_size // tile_size) + tile_column
                y0 = top + tile_row * tile_size
                x0 = left + tile_column * tile_size
                frame[y0 : y0 + tile_size, x0 : x0 + tile_size] = tile_palette[tile_index]

        frames.append(frame)
        centers_xy.append((center_x, center_y))

    return np.stack(frames), np.asarray(centers_xy, dtype=np.float32)


clip_frames, object_centers_xy = generate_clip()
assert clip_frames.shape == (FRAME_COUNT, FRAME_SIZE, FRAME_SIZE, 3)
assert clip_frames.dtype == np.uint8
print(f"Clip shape: {clip_frames.shape}; dtype: {clip_frames.dtype}; seed: {SEED}")
print(f"First and last object centers (x, y): {object_centers_xy[0].tolist()}, {object_centers_xy[-1].tolist()}")


In [ ]:
show_indices = [0, 4, 8, 12, 15]
fig, axes = plt.subplots(1, len(show_indices), figsize=(15, 3.2))
for axis, frame_index in zip(axes, show_indices):
    axis.imshow(clip_frames[frame_index])
    center = object_centers_xy[frame_index].astype(int).tolist()
    axis.set_title(f"Frame {frame_index} (center={center})")
    axis.axis("off")
fig.suptitle("Synthetic clip: the same patterned object moves across the frame")
fig.tight_layout()
plt.show()


## 2. Load the official model (optional)

Only run the model path after reviewing the source terms and setting `JEPA_LOAD_VJEPA21=1`. This downloads the pinned source code and the official checkpoint if they are not already cached.

The official Hub entrypoint returns an encoder and predictor. This lesson uses the encoder only. The encoder's published checkpoint stores its weights under `ema_encoder`; the prefix cleanup below follows the pinned upstream loader and uses a strict state-dict check.


In [ ]:
if LOAD_OFFICIAL_MODEL:
    try:
        import torch
        import torch.nn.functional as torch_functional
    except ImportError as error:
        raise RuntimeError(
            "The model path needs PyTorch. Install the suitable PyTorch wheel and the vjepa21 optional extra first."
        ) from error

    cpu_threads = max(1, int(os.environ.get("JEPA_CPU_THREADS", "4")))
    torch.set_num_threads(cpu_threads)

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    def available_memory_gib():
        try:
            import psutil
            return psutil.virtual_memory().available / (1024**3)
        except ImportError:
            pass
        if Path("/proc/meminfo").exists():
            for line in Path("/proc/meminfo").read_text().splitlines():
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) * 1024 / (1024**3)
        return None

    def process_peak_memory_gib():
        try:
            import resource
            value = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            if sys.platform == "darwin":
                return value / (1024**3)
            return value * 1024 / (1024**3)
        except (ImportError, AttributeError):
            return None

    available_memory_before_load = available_memory_gib()
    print(f"Device: {device}; CPU threads: {cpu_threads}")
    print(f"Available system memory before load (GiB): {available_memory_before_load}")
    print(f"Official code revision: {CODE_REVISION}")
    print(f"Checkpoint URL: {CHECKPOINT_URL}")
    print(f"Expected checkpoint bytes from source response: {EXPECTED_CHECKPOINT_BYTES:,}")
    print("Checkpoint terms: not separately specified in the official README; check current terms before use.")

    load_start = time.perf_counter()
    encoder, predictor = torch.hub.load(
        MODEL_REPO,
        MODEL_ENTRYPOINT,
        source="github",
        pretrained=False,
        num_frames=FRAME_COUNT,
        trust_repo=True,
    )
    del predictor
    gc.collect()

    preprocessor = torch.hub.load(
        MODEL_REPO,
        "vjepa2_preprocessor",
        source="github",
        pretrained=False,
        crop_size=FRAME_SIZE,
        trust_repo=True,
    )

    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint_state = torch.hub.load_state_dict_from_url(
        CHECKPOINT_URL,
        model_dir=str(CHECKPOINT_DIR),
        map_location="cpu",
        progress=True,
        weights_only=True,
    )
    if "ema_encoder" not in checkpoint_state:
        raise KeyError("The official checkpoint does not contain the expected ema_encoder key.")

    encoder_state = {
        key.replace("module.", "").replace("backbone.", ""): value
        for key, value in checkpoint_state["ema_encoder"].items()
    }
    encoder.load_state_dict(encoder_state, strict=True)
    del encoder_state, checkpoint_state
    gc.collect()

    encoder = encoder.to(device).eval()
    processed_views = preprocessor(clip_frames)
    assert len(processed_views) == 1, "Expected one deterministic evaluation view."
    video_tensor = processed_views[0].unsqueeze(0).to(device)
    expected_input = (1, 3, FRAME_COUNT, FRAME_SIZE, FRAME_SIZE)
    assert tuple(video_tensor.shape) == expected_input, (tuple(video_tensor.shape), expected_input)

    actual_bytes = CHECKPOINT_PATH.stat().st_size
    sha256 = hashlib.sha256()
    with CHECKPOINT_PATH.open("rb") as checkpoint_file:
        for chunk in iter(lambda: checkpoint_file.read(8 * 1024 * 1024), b""):
            sha256.update(chunk)

    run_metadata = {
        "status": "loaded",
        "model": MODEL_ENTRYPOINT,
        "code_revision": CODE_REVISION,
        "checkpoint_url": CHECKPOINT_URL,
        "checkpoint_bytes": actual_bytes,
        "checkpoint_sha256": sha256.hexdigest(),
        "checkpoint_terms": "not separately specified in official README; check current terms",
        "device": str(device),
        "input_shape": list(video_tensor.shape),
        "preprocessing": "pinned vjepa2_preprocessor(crop_size=384), evaluation resize, center crop, tensor conversion, ImageNet normalization",
        "model_load_seconds": round(time.perf_counter() - load_start, 3),
        "available_memory_before_load_gib": available_memory_before_load,
        "process_peak_memory_after_load_gib": process_peak_memory_gib(),
        "cpu_threads": cpu_threads,
    }
    print(json.dumps(run_metadata, indent=2))
else:
    print("Skipped: no official code or checkpoint was loaded. The clip above is the complete no-download path.")


## 3. Extract and inspect dense features

The encoder returns one vector per spatiotemporal patch. For a 16-frame input with 2-frame tubelets and a 384-pixel image with 16-pixel patches, the expected token layout is `batch × 8 × 24 × 24 × channels`.

The plotted value is the L2 norm of each feature vector. It is a view of the representation, not a task score.


In [ ]:
if encoder is not None:
    inference_start = time.perf_counter()
    with torch.inference_mode():
        encoded_tokens = encoder(video_tensor)
    inference_seconds = time.perf_counter() - inference_start

    if isinstance(encoded_tokens, (tuple, list)):
        raise TypeError(f"Expected one token tensor from the encoder, got {type(encoded_tokens).__name__}.")
    expected_tokens = TIME_GROUPS * PATCH_ROWS * PATCH_COLUMNS
    assert encoded_tokens.ndim == 3, tuple(encoded_tokens.shape)
    assert encoded_tokens.shape[0] == 1, tuple(encoded_tokens.shape)
    assert encoded_tokens.shape[1] == expected_tokens, (tuple(encoded_tokens.shape), expected_tokens)
    assert torch.isfinite(encoded_tokens).all().item(), "Encoder output contains non-finite values."

    channel_count = encoded_tokens.shape[-1]
    feature_grid = encoded_tokens.reshape(
        1, TIME_GROUPS, PATCH_ROWS, PATCH_COLUMNS, channel_count
    )
    feature_norms = feature_grid[0].norm(dim=-1).float().cpu().numpy()
    run_metadata.update(
        {
            "feature_shape_batch_time_rows_columns_channels": list(feature_grid.shape),
            "feature_channels": int(channel_count),
            "inference_seconds": round(inference_seconds, 3),
            "available_memory_after_inference_gib": available_memory_gib(),
            "process_peak_memory_after_inference_gib": process_peak_memory_gib(),
        }
    )
    print(f"Feature grid: {tuple(feature_grid.shape)}")
    print(f"Encoder inference time (seconds): {inference_seconds:.3f}")
    print(f"Available system memory after inference (GiB): {run_metadata['available_memory_after_inference_gib']}")
    print(f"Process peak RSS after inference (GiB): {run_metadata['process_peak_memory_after_inference_gib']}")
else:
    print("Skipped: enable the official model path to extract learned features.")


In [ ]:
if feature_grid is not None:
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    for time_group, axis in enumerate(axes.flat):
        image_index = min(time_group * TUBELET_SIZE, FRAME_COUNT - 1)
        axis.imshow(feature_norms[time_group], cmap="magma")
        axis.set_title(f"Tubelet {time_group} (starts at frame {image_index})")
        axis.set_xticks([0, 5, 11, 17, 23])
        axis.set_yticks([0, 5, 11, 17, 23])
        axis.set_xlabel("patch column")
        axis.set_ylabel("patch row")
    fig.suptitle("L2 norm of each dense encoder feature")
    fig.tight_layout()
    plt.show()


## 4. Compare temporal patch correspondence

For a query tubelet and a later target tubelet, this section finds the most similar patch across the full later grid. It reports the retrieved patch and its distance from the known object-center patch.

The baseline is the mean RGB value in each patch, averaged across the same two-frame tubelet. Both methods use cosine similarity. A miss is a measured result; this one synthetic clip cannot establish general tracking ability.


In [ ]:
if feature_grid is not None:
    QUERY_GROUP = 1
    TARGET_GROUP = 6
    NORMALIZE_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    NORMALIZE_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    # The pinned transform resizes the short side to int(384 * 256 / 224) and center-crops to 384.
    RESIZED_SHORT_SIDE = int(FRAME_SIZE * 256 / 224)
    RESIZE_SCALE = RESIZED_SHORT_SIDE / FRAME_SIZE
    CROP_OFFSET = (RESIZED_SHORT_SIDE - FRAME_SIZE) / 2

    def object_patch_for_group(group_index):
        center_xy = object_centers_xy[
            group_index * TUBELET_SIZE : (group_index + 1) * TUBELET_SIZE
        ].mean(axis=0)
        transformed_xy = center_xy * RESIZE_SCALE - CROP_OFFSET
        column = int(np.clip(np.floor(transformed_xy[0] / PATCH_SIZE), 0, PATCH_COLUMNS - 1))
        row = int(np.clip(np.floor(transformed_xy[1] / PATCH_SIZE), 0, PATCH_ROWS - 1))
        return row, column

    query_row, query_column = object_patch_for_group(QUERY_GROUP)
    target_row, target_column = object_patch_for_group(TARGET_GROUP)

    query_feature = feature_grid[0, QUERY_GROUP, query_row, query_column]
    later_features = feature_grid[0, TARGET_GROUP].reshape(-1, channel_count)
    feature_similarities = torch_functional.cosine_similarity(
        query_feature.unsqueeze(0), later_features, dim=-1
    )
    feature_retrieved_flat = int(feature_similarities.argmax().item())
    feature_retrieved_row, feature_retrieved_column = divmod(feature_retrieved_flat, PATCH_COLUMNS)

    # Undo the official ImageNet normalization so the baseline uses processed RGB values.
    normalized_video = video_tensor[0].detach().cpu()
    mean_tensor = torch.tensor(NORMALIZE_MEAN).view(3, 1, 1, 1)
    std_tensor = torch.tensor(NORMALIZE_STD).view(3, 1, 1, 1)
    processed_rgb = (normalized_video * std_tensor + mean_tensor).clamp(0, 1)
    rgb_groups = processed_rgb.reshape(
        3, TIME_GROUPS, TUBELET_SIZE, FRAME_SIZE, FRAME_SIZE
    ).mean(dim=2).permute(1, 2, 3, 0)

    def rgb_patch_descriptors(time_group):
        grouped_rgb = rgb_groups[time_group]
        return grouped_rgb.reshape(
            PATCH_ROWS, PATCH_SIZE, PATCH_COLUMNS, PATCH_SIZE, 3
        ).mean(dim=(1, 3)).reshape(-1, 3)

    query_rgb = rgb_patch_descriptors(QUERY_GROUP)[query_row * PATCH_COLUMNS + query_column]
    later_rgb = rgb_patch_descriptors(TARGET_GROUP)
    pixel_similarities = torch_functional.cosine_similarity(
        query_rgb.unsqueeze(0), later_rgb, dim=-1
    )
    pixel_retrieved_flat = int(pixel_similarities.argmax().item())
    pixel_retrieved_row, pixel_retrieved_column = divmod(pixel_retrieved_flat, PATCH_COLUMNS)

    target_patch = np.array([target_row, target_column])
    feature_retrieved_patch = np.array([feature_retrieved_row, feature_retrieved_column])
    pixel_retrieved_patch = np.array([pixel_retrieved_row, pixel_retrieved_column])

    correspondence = {
        "query_group": QUERY_GROUP,
        "target_group": TARGET_GROUP,
        "known_target_patch_row_column": target_patch.tolist(),
        "vjepa": {
            "retrieved_patch_row_column": feature_retrieved_patch.tolist(),
            "cosine_similarity": float(feature_similarities[feature_retrieved_flat].item()),
            "distance_from_known_target_patches": float(np.linalg.norm(feature_retrieved_patch - target_patch)),
        },
        "mean_rgb_baseline": {
            "retrieved_patch_row_column": pixel_retrieved_patch.tolist(),
            "cosine_similarity": float(pixel_similarities[pixel_retrieved_flat].item()),
            "distance_from_known_target_patches": float(np.linalg.norm(pixel_retrieved_patch - target_patch)),
        },
        "metric": "cosine similarity over all 24×24 patches in the later tubelet",
        "interpretation": "one synthetic-clip measurement; no pass threshold or general model claim",
    }
    run_metadata["correspondence"] = correspondence
    print(json.dumps(correspondence, indent=2))

    target_frame = rgb_groups[TARGET_GROUP].numpy()
    fig, axis = plt.subplots(figsize=(6, 6))
    axis.imshow(target_frame)
    for patch, color, label in [
        (target_patch, "lime", "known target"),
        (feature_retrieved_patch, "cyan", "V-JEPA retrieval"),
        (pixel_retrieved_patch, "white", "mean-RGB retrieval"),
    ]:
        row, column = patch
        axis.add_patch(
            plt.Rectangle(
                (column * PATCH_SIZE, row * PATCH_SIZE),
                PATCH_SIZE,
                PATCH_SIZE,
                fill=False,
                edgecolor=color,
                linewidth=2,
                label=label,
            )
        )
    axis.legend(loc="upper right")
    axis.set_title(f"Target tubelet {TARGET_GROUP}: known and retrieved patches")
    axis.set_xlim(0, FRAME_SIZE)
    axis.set_ylim(FRAME_SIZE, 0)
    axis.set_xlabel("pixel column")
    axis.set_ylabel("pixel row")
    plt.show()
else:
    print("Skipped: correspondence needs the official feature grid.")


## 5. Read the result carefully

The feature plot and correspondence numbers describe this checkpoint, this preprocessing, this generated clip, and this machine. They do not reproduce the paper's Ego4D, EPIC-KITCHENS, robotics, depth, or recognition evaluations.

When you share a local run, include the printed JSON record. Keep the checkpoint, generated clip, and derived features out of version control. For paper-level reproduction work, use the separate [paper-reproduction matrix](../docs/REPRODUCTION_MATRIX.md) and [evaluation protocol](../docs/EVALUATION_PROTOCOL.md).
